In [ ]:
import os
os.getcwd()

In [ ]:
# Share_point = r'C:\Users\giovanni.sgaravatti\Bruegel\Research - 2021-11 European natural gas imports\Code\Imports'
Share_point = r'C:\Users\ugne.keliauskaite\Bruegel\Research - 2021-11 European natural gas imports\Code\Imports'

In [ ]:
os.chdir(Share_point)

In [ ]:
import json
import requests
import pandas as pd
import numpy as np

from datetime import datetime
from datetime import date, timedelta
from dateutil.relativedelta import relativedelta

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.dates import DateFormatter
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import seaborn as sns

import eurostat  # python wrapper for taking data.
import time

In [ ]:
# Converter for metric tonnes to M3m 
# converter = -0.001397
converter = -0.00138

### In Million tonnes

In [ ]:
d1 = pd.read_excel('raw_data/LNG/Europe LNG 2017_2018.xlsx',index_col=1)
d2 = pd.read_excel('raw_data/LNG/Europe LNG 2019_2020.xlsx',index_col=1)
d3 = pd.read_excel('raw_data/LNG/Europe LNG 2021_2022.xlsx',index_col=1)
d4 = pd.read_excel('raw_data/LNG/Europe LNG 2023.xlsx',index_col=1)
d5 = pd.read_excel('raw_data/LNG/Europe LNG 2024.xlsx',index_col=1)
d6 = pd.read_excel('raw_data/LNG/Europe LNG 2025.xlsx',index_col=1)
d7 = pd.read_excel('raw_data/LNG/Europe LNG 2026.xlsx',index_col=1)


d7 = d7.groupby(['Port_Country', 'Origin_Berthcountry']).sum().reset_index()


In [ ]:
# Merge 2017 to 2020
df_20 = pd.merge(d1,d2,on=['Origin_Berthcountry','Port_Country'],how='outer')

In [ ]:
# Merge 2017 to 2022
df_22 = pd.merge(df_20,d3,on=['Origin_Berthcountry','Port_Country'],how='outer')

In [ ]:
#df_22
#d4

In [ ]:
# Merge 2022 to 2023
df_23 = pd.merge(df_22,d4,on=['Origin_Berthcountry','Port_Country'],how='outer')

In [ ]:
# Merge 2023 to 2024
df_24 = pd.merge(df_23,d5,on=['Origin_Berthcountry','Port_Country'],how='outer')

In [ ]:
# Merge 2024 to 2025
df_25 = pd.merge(df_24,d6,on=['Origin_Berthcountry','Port_Country'],how='outer')

In [ ]:
# Merge 2025 to 2026
df_26 = pd.merge(df_25,d7,on=['Origin_Berthcountry','Port_Country'],how='outer')

In [ ]:
not_EU = ['United Kingdom','Turkey','Russia']
df_EU27 = df_26.loc[~df_26['Port_Country'].isin((not_EU))] 

In [ ]:
# you can void running it
df_EU27

In [ ]:
## Here we get 
dff = df_EU27.groupby(['Origin_Berthcountry']).sum(numeric_only=True)
dff


In [ ]:
dff_T = dff.T #drop(['nan_x','nan_y'])
dff_T

In [ ]:
dff_T.index = pd.to_datetime(dff_T.index)
dff_T = dff_T.sort_index()
dff_T

In [ ]:
# check EU LNG imports in 2022
# dff_T['2022'].sum().sum()/1000000

### in M3m

In [ ]:
dff_M3m = dff_T * converter

In [ ]:
# Remove European re-exports
del dff_M3m['Belgium']
del dff_M3m['France']
del dff_M3m['Lithuania']
del dff_M3m['Netherlands']
del dff_M3m['Spain']
del dff_M3m['Germany']
del dff_M3m['Finland']

In [ ]:
dff_EU = dff_M3m


In [ ]:
dff_M3m.columns

In [ ]:
dff_M3m.tail()

In [ ]:
# look only at the big exporters to the EU
include = ['Algeria','Norway','Nigeria','Qatar','Russia','Trinidad & Tobago','United States of America','Egypt']
exclude = list(set(dff_M3m.columns)-set(include))

In [ ]:
## Define an "Other" category
dff_M3m['Other']= dff_M3m[exclude].sum(axis=1)

In [ ]:
# dff_M3m['Argentina'].loc['2020':]+dff_M3m['Brazil'].loc['2020':]+dff_M3m['Peru'].loc['2020':]

In [ ]:
# os.chdir(r'C:\Users\giovanni.sgaravatti\Bruegel\Research - 2021-11 European natural gas imports\Data\LNG') # Gio
os.chdir(r'C:\Users\ugne.keliauskaite\Bruegel\Research - 2021-11 European natural gas imports\Data\LNG') # Ugne

In [ ]:
dff_M3m['month'] = dff_M3m.index.strftime("%m")


In [ ]:
os.getcwd()

In [ ]:
dff_M3m

In [ ]:
### From country of origin
data = pd.DataFrame()
for country in ['Algeria','Norway','Nigeria','Qatar','Russia','Trinidad & Tobago','United States of America','Egypt','Other']:

    ctemp = dff_M3m[country][:'2026-01-31'] # to update to the last month available
    ctemp=ctemp.to_frame()
    ctemp['month'] = ctemp.index.strftime("%m")
    
#     minvalues = ctemp.loc['2017':'2020'].groupby('month').min()
#     maxvalues = ctemp.loc['2017':'2020'].groupby('month').max()
    avgvalues = ctemp.loc['2017':'2021'].groupby('month').mean()   
#     values2021 = ctemp.loc['2021'].groupby('month').mean()   
    values2022 = ctemp.loc['2022'].groupby('month').max()
    values2023 = ctemp.loc['2023'].groupby('month').max()
    values2024 = ctemp.loc['2024'].groupby('month').max()
    values2025 = ctemp.loc['2025'].groupby('month').max()
    values2026 = ctemp.loc['2026'].groupby('month').max()

    fig, ax = plt.subplots(figsize=(8, 8))

    ax.plot(avgvalues,label='Average 2017-2021')
#     ax.plot(values2021,label='2021')
    ax.plot(values2022,label='2022')
    ax.plot(values2023,label='2023')
    ax.plot(values2024,label='2024')
    ax.plot(values2025,label='2025')
    ax.plot(values2025,label='2026')


    plt.title('EU27 LNG imports in M3m {}'.format(country),fontsize='16')
    plt.ylabel('M3M',fontsize=12,rotation=0,labelpad=25)
    plt.xlabel('Month Number',fontsize=12,rotation=0)
    plt.legend()
    plt.show()
    fig.savefig('pictures/{}.png'.format(country),dpi=300,bbox_inches="tight")

#   data['{}_min'.format(country)] = minvalues
#   data['{}_max'.format(country)] = maxvalues
    data['{}_avg'.format(country)] = avgvalues
#   data['{}_2021'.format(country)] = values2021 
    data['{}_2022'.format(country)] = values2022
    data['{}_2023'.format(country)] = values2023
    data['{}_2024'.format(country)] = values2024
    data['{}_2025'.format(country)] = values2025
    data['{}_2026'.format(country)] = values2026

In [ ]:
today = date.today()    
data = data.round(1)
data.to_csv('LNG_imports_avg_{}.csv'.format(today))


#### Get only 2019 onwards values

In [ ]:
df_late =dff_M3m.loc['2019':]

In [ ]:
df_late=df_late.rename(columns={'United States of America':'United States'})

In [ ]:
df_late=df_late.drop('month',axis=1)
df_late = df_late[['Qatar','Algeria','Nigeria','Russia','Norway','United States','Trinidad & Tobago','Egypt','Other']]

In [ ]:
import os
os.getcwd()

In [ ]:
df_late.to_excel(r'Bloomberg/granular LNG imports.xlsx')

#### Aggregate to avoid problems with Bloomberg

In [ ]:
dff_EU = dff_EU.loc['2019':]

In [ ]:
dff_EU.columns


In [ ]:
# dff_EU was defined before playing around with include/exlude in dff_M3m (feel free to change)

In [ ]:
exclude_2 = ['Argentina','Australia','Brazil','China','Indonesia','Jamaica','Malaysia','Peru','United Kingdom','Singapore','South Korea']

In [ ]:
set(dff_EU.columns)-set(exclude_2)

In [ ]:
dff_EU.columns

In [ ]:
dff_EU['Other']=dff_EU[exclude_2].sum(axis=1)

In [ ]:
dff_EU

In [ ]:
dff_EU['America'] = dff_EU['United States of America']+dff_EU['Trinidad & Tobago']+dff_EU['Dominican Republic']
dff_EU['Africa'] = dff_EU['Algeria']+dff_EU['Angola']+dff_EU['Nigeria']+dff_EU['Egypt']+dff_EU['Cameroon']+dff_EU['Equatorial Guinea']+dff_EU['Mozambique']
dff_EU['Middle East'] = dff_EU['Qatar']+dff_EU['Oman']+dff_EU['United Arab Emirates']
dff_EU['Other']=dff_EU['Other']+dff_EU['Norway']

In [ ]:
dff_EU.head()

In [ ]:
df_plot = dff_EU[['America','Africa','Middle East','Russia','Other']]

In [ ]:
df_plot.head()

In [ ]:
df_plot['dates'] = df_plot.index.strftime('%m/%Y')

In [ ]:
os.getcwd()

In [ ]:
today

In [ ]:
df_plot.to_excel(r'Bloomberg\LNG plot data {}.xlsx'.format(today))

In [ ]:
x = df_plot['dates']
width = 0.80       # the width of the bars: can also be len(x) sequence

fig, ax = plt.subplots(figsize=(15,10))

ax.bar(x, df_plot['America'], width, label='America')
ax.bar(x, df_plot['Africa'], width, bottom=df_plot['America'],
       label='Africa')
ax.bar(x, df_plot['Middle East'], width, bottom=df_plot['America']+df_plot['Africa'],
       label='Middle East',color='gray')
ax.bar(x, df_plot['Russia'], width, bottom=df_plot['America']+df_plot['Africa']+df_plot['Middle East'],
       label='Russia',color='red')
ax.bar(x, df_plot['Other'], width, bottom=df_plot['America']+df_plot['Africa']+df_plot['Middle East']+df_plot['Russia'],
       label='Other',color='black')

fig.set_facecolor('white')

plt.setp(ax.get_xticklabels(), rotation=60, horizontalalignment='right')

ax.set_ylabel('M3m')
ax.set_title('LNG imports in the EU by world region')
ax.legend()

plt.show()

### In bcm

In [ ]:
df_late.columns

In [ ]:
df_bcm = df_late/1000

In [ ]:
df_bcm = df_bcm['2021-09':]

In [ ]:
df_bcm.plot(figsize=(15, 9),kind='area',ylabel='bcm', title='LNG imports in the EU by supplier')

In [ ]:
df_twh = df_bcm*10.3 

In [ ]:
df_twh = df_twh[['United States','Russia', 'Qatar',  'Norway', 'Algeria', 'Nigeria', 'Trinidad & Tobago', 'Egypt', 'Other']]

In [ ]:
colors = {'United States':'#0080C7','Russia':'#E67425', 'Qatar':'#FFC000',  'Norway':'black', 'Algeria':'orange', 'Nigeria':'#2BB0CB', 'Trinidad & Tobago':'#0080C7', 'Egypt':'blue', 'Other':'#58A944'}

In [ ]:
df_twh.plot(figsize=(15, 9),kind='area',ylabel='bcm', title='LNG imports in the EU by supplier',color=colors)

### Checks against the Quarterly report of the Commission

In [ ]:
df_bcm['M_Y'] = df_bcm.index.strftime('%m_%Y')

In [ ]:
df_bcm.plot(x='M_Y', kind='bar', stacked=True,
        title='LNG imports in the EU by supplier',figsize=(18, 9),
           ylabel='bcm', xlabel='month')

In [ ]:
from IPython.display import Image
Image("Commission Qreport 2023 by source.png")

In the second quarter of 2022, the United States proved to be the biggest LNG supplier of the EU, by a large margin to its competitors, ensuring 16 bcm of the EU LNG imports within a single quarter (for comparison: EU LNG imports from the US amounted to 22 bcm in 2021 as whole), representing around 45% of the total imports. Year-on-year, LNG imports from the US were up by 117%, and the share of the US in total EU LNG imports rose by 14 percentage points. In 2022 so far, the EU imported around 39 bcm LNG from the US, implying that the objective of the March 2022 EU-US joint statement on energy security12, which foresaw and increase of 15 bcm compared to 2021, has already been fulfilled.

In [ ]:
df_bcm['United States'].loc['2022-04':'2022-06'].sum()

In [ ]:
df_bcm['United States'].loc['2021'].sum()

In [ ]:
df_bcm['United States'].loc['2022-01':'2022-08'].sum()

In spite of ongoing geopolitical tensions, Russia remained the second biggest LNG supplier of the EU, representing 18% (6.5 bcm, up by 28% year-on-year). Qatar was the third most important EU LNG source (with an import share of 13% and imports amounting to 4.6 bcm, +7%), followed by Nigeria on the fourth place (with an import share of only 8% - 2.7 bcm, but falling by 11% year-on-year). LNG imports from Algeria amounted to 2.1 bcm, falling by 12% year-on-year and representing only 6% of the total imports. Within other sources, LNG imports from Egypt rose almost five-fold year-on-year, reaching 1.3 bcm (4% of the total EU imports), while those from Trinidad and Tobago amounted to 0.9 bcm and ensured around 3% of the total EU LNG imports – See Figure 16

In [ ]:
df_bcm['Russia'].loc['2022-04':'2022-06'].sum()

In [ ]:
df_bcm['Qatar'].loc['2022-04':'2022-06'].sum()

In [ ]:
df_bcm['Nigeria'].loc['2022-04':'2022-06'].sum()

In [ ]:
df_bcm['Algeria'].loc['2022-04':'2022-06'].sum()

In [ ]:
dff_M3m['Egypt'].loc['2022-04':'2022-06'].sum() # in M3m

In [ ]:
df_bcm['Trinidad & Tobago'].loc['2022-04':'2022-06'].sum()

In Q2 2022, Norway still had a very low share (around 0.3%) in total EU LNG imports, however, in June the Hammerfest LNG liquefaction terminal restarted its operation, being under repair and maintenance works since the fire incident at the end of September 2020 (See more in Chapter 1.4) and is expected to measurably contribute to the EU LNG supply as of from the second half of 2022.

In [ ]:
N=df_bcm['Norway'].loc['2022-04':'2022-06'].sum()
T=df_bcm[['Qatar','Algeria','Nigeria','Russia','Norway','United States','Trinidad & Tobago','Egypt','Other']].loc['2022-04':'2022-06'].sum()

In [ ]:
TT= T.sum()

In [ ]:
N/TT*100

### Single-out Spain and Portugal

In [ ]:
df_tot = df_EU27.reset_index()

In [ ]:
iberia = df_tot[df_tot['Port_Country'].isin(['Spain','Portugal'])]

In [ ]:
cols = iberia.iloc[:,:2]

In [ ]:
# convert to M3m
values = iberia.iloc[:,2:]*converter

In [ ]:
iberia = pd.concat([cols,values], axis=1)

In [ ]:
iberia.to_excel(r'C:\Users\giovanni.sgaravatti\Bruegel\Research - 2023-04 How the EU can phase out Russian LNG\Data\Iberian LNG by source.xlsx')